In [257]:
import pandas as pd

import sqlite3

conn = sqlite3.connect('../cafef.db')

In [258]:
df_bs = pd.read_sql("Select * from fireant_balance_sheet", conn, params=[])
df_is = pd.read_sql("Select * from fireant_income_statement", conn, params=[])
df_cf = pd.read_sql("Select * from fireant_cash_flow", conn, params=[])
df_cf_dr = pd.read_sql("Select * from fireant_cash_flow_direct", conn, params=[])

In [259]:
df_bs["isNA"] = df_bs["value"].isna()
df_bs

,symbol,section,code,name,level,period,value,isNA
0,A32,TÀI SẢN,1,TÀI SẢN,1,2016,NaN,True
1,A32,TÀI SẢN,101,A. Tài sản lưu động và đầu tư ngắn hạn,2,2016,3.432787e+11,False
2,A32,TÀI SẢN,10101,I. Tiền và các khoản tương đương tiền,3,2016,1.397735e+11,False
3,A32,TÀI SẢN,1010101,1. Tiền,4,2016,1.006804e+10,False
4,A32,TÀI SẢN,1010102,2. Các khoản tương đương tiền,4,2016,1.297055e+11,False
...,...,...,...,...,...,...,...,...
2726227,THA,NGUỒN VỐN,30202,II. Nguồn kinh phí và quỹ khác,3,2016,0.000000e+00,False
2726228,THA,NGUỒN VỐN,3020201,1. Nguồn kinh phí,4,2016,0.000000e+00,False
2726229,THA,NGUỒN VỐN,3020202,2. Nguồn kinh phí đã hình thành tài sản cố định,4,2016,0.000000e+00,False
2726230,THA,NGUỒN VỐN,3020203,3. Quỹ dự phòng trợ cấp mất việc làm,4,2016,0.000000e+00,False


In [262]:
non_financial_firms = pd.read_csv("../data/preprocessing_pipeline_results/simple_filter_fa_price.csv")
non_financial_firms["symbol"]

0       TNG
1       L18
2       NTL
3       NHC
4       DHA
       ... 
1809    DKG
1810    HTH
1811    VLS
1812    TRV
1813    HHB
Name: symbol, Length: 1814, dtype: str

In [263]:
import statement_fields as sf
import importlib
importlib.reload(sf)
SRC = sf.FIREANT
frames = {
    sf.BALANCE_SHEET:    df_bs,
    sf.INCOME_STATEMENT: df_is,
    sf.CASH_FLOW:        df_cf,
    sf.CASH_FLOW_DIRECT: df_cf_dr,
}

def build_panel(frames, source=SRC):
    parts = []
    for stmt in (sf.BALANCE_SHEET, sf.INCOME_STATEMENT, sf.CASH_FLOW, sf.CASH_FLOW_DIRECT):
        keys = [k for s, k in sf.REQUIRED if s == stmt]
        parts.append(sf.widen(frames[stmt], source, stmt, keys))
    panel = pd.concat(parts, axis=1)
    return panel

panel = build_panel(frames)
panel["cfo"] = panel["cfo"].fillna(panel["cfo_direct"])
panel["cfi"] = panel["cfi"].fillna(panel["cfi_direct"])
panel["cff"] = panel["cff"].fillna(panel["cff_direct"])
panel["cash_begin"] = panel["cash_begin"].fillna(panel["cash_begin_direct"])
panel["cash_end"] = panel["cash_end"].fillna(panel["cash_end_direct"])
panel["net_cash_flow"] = panel["net_cash_flow"].fillna(panel["net_cash_flow_direct"])
panel["fx_effect"] = panel["fx_effect"].fillna(panel["fx_effect_direct"])
panel

owner_equity  minority_interest  paid_in_capital  \
symbol period                                                     
A32    2016    1.628425e+11                0.0     6.800000e+10   
       2017    1.763000e+11                0.0     6.800000e+10   
       2018    2.008266e+11                0.0     6.800000e+10   
       2019    2.236159e+11                0.0     6.800000e+10   
       2020    2.422229e+11                0.0     6.800000e+10   
...                     ...                ...              ...   
VTR    2016             NaN                NaN              NaN   
VW3    2016             NaN                NaN              NaN   
VXP    2015             NaN                NaN              NaN   
X18    2010             NaN                NaN              NaN   
VEE    2009             NaN                NaN              NaN   

               treasury_stock  total_assets  long_term_debt  current_assets  \
symbol period                                                                 
A32    2016               0.0  4.650048e+11    2.429378e+09    3.432787e+11   
       2017               0.0  5.009923e+11    2.429378e+09    3.741267e+11   
       2018               0.0  4.688411e+11    2.429378e+09    3.358630e+11   
       2019               0.0  4.349303e+11    1.429378e+09    2.987680e+11   
       2020               0.0  4.882955e+11    0.000000e+00    3.566913e+11   
...                       ...           ...             ...             ...   
VTR    2016               NaN           NaN             NaN             NaN   
VW3    2016               NaN           NaN             NaN             NaN   
VXP    2015               NaN           NaN             NaN             NaN   
X18    2010               NaN           NaN             NaN             NaN   
VEE    2009               NaN           NaN             NaN             NaN   

               current_liabilities  total_liabilities  equity_section  ...  \
symbol period                                                          ...   
A32    2016           2.991735e+11       3.016029e+11    1.634019e+11  ...   
       2017           3.222796e+11       3.247090e+11    1.762833e+11  ...   
       2018           2.656000e+11       2.680293e+11    2.008118e+11  ...   
       2019           2.098882e+11       2.113176e+11    2.236127e+11  ...   
       2020           2.460785e+11       2.460785e+11    2.422169e+11  ...   
...                            ...                ...             ...  ...   
VTR    2016                    NaN                NaN             NaN  ...   
VW3    2016                    NaN                NaN             NaN  ...   
VXP    2015                    NaN                NaN             NaN  ...   
X18    2010                    NaN                NaN             NaN  ...   
VEE    2009                    NaN                NaN             NaN  ...   

               net_cash_flow    cash_begin      cash_end    cfo_direct  \
symbol period                                                            
A32    2016    -1.336047e+11  2.733782e+11  1.397735e+11 -3.127942e+10   
       2017     5.884773e+09  1.397735e+11  1.456583e+11 -6.679452e+09   
       2018    -8.736751e+10  1.456583e+11  5.829081e+10  5.224003e+10   
       2019     2.222948e+09  5.829081e+10  6.051375e+10  1.120983e+10   
       2020    -1.610927e+10  6.051375e+10  4.435908e+10  4.279751e+09   
...                      ...           ...           ...           ...   
VTR    2016              NaN           NaN           NaN           NaN   
VW3    2016              NaN           NaN           NaN           NaN   
VXP    2015     1.877209e+10  7.524233e+09  2.629632e+10           NaN   
X18    2010    -2.292676e+10  2.809413e+10  5.167372e+09           NaN   
VEE    2009     7.711102e+09  4.473609e+09  1.218471e+10           NaN   

                 cfi_direct    cff_direct  fx_effect_direct  \
symbol period                                                 
A32    2016

In [264]:
previous = panel["paid_in_capital"].copy()

previous = previous.reset_index()
previous["period"] = previous["period"].astype(int) + 1
previous["period"] = previous["period"].astype(str)
previous.set_index(keys=["symbol", "period"], inplace=True)
previous

paid_in_capital
symbol period                 
A32    2017       6.800000e+10
       2018       6.800000e+10
       2019       6.800000e+10
       2020       6.800000e+10
       2021       6.800000e+10
...                        ...
VTR    2017                NaN
VW3    2017                NaN
VXP    2016                NaN
X18    2011                NaN
VEE    2010                NaN

[23609 rows x 1 columns]

In [265]:
panel = panel.join(previous, how="left", on=["symbol", "period"], rsuffix="_prev")
panel["d_paid_in_capital"] = (panel["paid_in_capital"] - panel["paid_in_capital_prev"]).abs()
panel

owner_equity  minority_interest  paid_in_capital  \
symbol period                                                     
A32    2016    1.628425e+11                0.0     6.800000e+10   
       2017    1.763000e+11                0.0     6.800000e+10   
       2018    2.008266e+11                0.0     6.800000e+10   
       2019    2.236159e+11                0.0     6.800000e+10   
       2020    2.422229e+11                0.0     6.800000e+10   
...                     ...                ...              ...   
VTR    2016             NaN                NaN              NaN   
VW3    2016             NaN                NaN              NaN   
VXP    2015             NaN                NaN              NaN   
X18    2010             NaN                NaN              NaN   
VEE    2009             NaN                NaN              NaN   

               treasury_stock  total_assets  long_term_debt  current_assets  \
symbol period                                                                 
A32    2016               0.0  4.650048e+11    2.429378e+09    3.432787e+11   
       2017               0.0  5.009923e+11    2.429378e+09    3.741267e+11   
       2018               0.0  4.688411e+11    2.429378e+09    3.358630e+11   
       2019               0.0  4.349303e+11    1.429378e+09    2.987680e+11   
       2020               0.0  4.882955e+11    0.000000e+00    3.566913e+11   
...                       ...           ...             ...             ...   
VTR    2016               NaN           NaN             NaN             NaN   
VW3    2016               NaN           NaN             NaN             NaN   
VXP    2015               NaN           NaN             NaN             NaN   
X18    2010               NaN           NaN             NaN             NaN   
VEE    2009               NaN           NaN             NaN             NaN   

               current_liabilities  total_liabilities  equity_section  ...  \
symbol period                                                          ...   
A32    2016           2.991735e+11       3.016029e+11    1.634019e+11  ...   
       2017           3.222796e+11       3.247090e+11    1.762833e+11  ...   
       2018           2.656000e+11       2.680293e+11    2.008118e+11  ...   
       2019           2.098882e+11       2.113176e+11    2.236127e+11  ...   
       2020           2.460785e+11       2.460785e+11    2.422169e+11  ...   
...                            ...                ...             ...  ...   
VTR    2016                    NaN                NaN             NaN  ...   
VW3    2016                    NaN                NaN             NaN  ...   
VXP    2015                    NaN                NaN             NaN  ...   
X18    2010                    NaN                NaN             NaN  ...   
VEE    2009                    NaN                NaN             NaN  ...   

                   cash_end    cfo_direct    cfi_direct    cff_direct  \
symbol period                                                           
A32    2016    1.397735e+11 -3.127942e+10 -8.319863e+09 -9.400540e+10   
       2017    1.456583e+11 -6.679452e+09 -2.375775e+09  1.494000e+10   
       2018    5.829081e+10  5.224003e+10  3.452162e+09 -1.430597e+11   
       2019    6.051375e+10  1.120983e+10  9.204970e+08 -9.907377e+09   
       2020    4.435908e+10  4.279751e+09  3.292742e+09 -2.368176e+10   
...                     ...           ...           ...           ...   
VTR    2016             NaN           NaN           NaN           NaN   
VW3    2016             NaN           NaN           NaN           NaN   
VXP    2015    2.629632e+10           NaN           NaN           NaN   
X18    2010    5.167372e+09           NaN           NaN           NaN   
VEE    2009    1.218471e+10           NaN           NaN           NaN   

               fx_effect_direct  net_cash_flow_direct  cash_begin_direct  \
symbol period                                                             

In [266]:
panel[panel["stock_issuance_proceeds"].isna()]

owner_equity  minority_interest  paid_in_capital  \
symbol period                                                     
A32    2016    1.628425e+11                0.0     6.800000e+10   
       2017    1.763000e+11                0.0     6.800000e+10   
       2018    2.008266e+11                0.0     6.800000e+10   
       2019    2.236159e+11                0.0     6.800000e+10   
       2020    2.422229e+11                0.0     6.800000e+10   
...                     ...                ...              ...   
VRE    2013             NaN                NaN              NaN   
VTK    2014             NaN                NaN              NaN   
VTP    2015             NaN                NaN              NaN   
VTR    2016             NaN                NaN              NaN   
VW3    2016             NaN                NaN              NaN   

               treasury_stock  total_assets  long_term_debt  current_assets  \
symbol period                                                                 
A32    2016               0.0  4.650048e+11    2.429378e+09    3.432787e+11   
       2017               0.0  5.009923e+11    2.429378e+09    3.741267e+11   
       2018               0.0  4.688411e+11    2.429378e+09    3.358630e+11   
       2019               0.0  4.349303e+11    1.429378e+09    2.987680e+11   
       2020               0.0  4.882955e+11    0.000000e+00    3.566913e+11   
...                       ...           ...             ...             ...   
VRE    2013               NaN           NaN             NaN             NaN   
VTK    2014               NaN           NaN             NaN             NaN   
VTP    2015               NaN           NaN             NaN             NaN   
VTR    2016               NaN           NaN             NaN             NaN   
VW3    2016               NaN           NaN             NaN             NaN   

               current_liabilities  total_liabilities  equity_section  ...  \
symbol period                                                          ...   
A32    2016           2.991735e+11       3.016029e+11    1.634019e+11  ...   
       2017           3.222796e+11       3.247090e+11    1.762833e+11  ...   
       2018           2.656000e+11       2.680293e+11    2.008118e+11  ...   
       2019           2.098882e+11       2.113176e+11    2.236127e+11  ...   
       2020           2.460785e+11       2.460785e+11    2.422169e+11  ...   
...                            ...                ...             ...  ...   
VRE    2013                    NaN                NaN             NaN  ...   
VTK    2014                    NaN                NaN             NaN  ...   
VTP    2015                    NaN                NaN             NaN  ...   
VTR    2016                    NaN                NaN             NaN  ...   
VW3    2016                    NaN                NaN             NaN  ...   

                   cash_end    cfo_direct    cfi_direct    cff_direct  \
symbol period                                                           
A32    2016    1.397735e+11 -3.127942e+10 -8.319863e+09 -9.400540e+10   
       2017    1.456583e+11 -6.679452e+09 -2.375775e+09  1.494000e+10   
       2018    5.829081e+10  5.224003e+10  3.452162e+09 -1.430597e+11   
       2019    6.051375e+10  1.120983e+10  9.204970e+08 -9.907377e+09   
       2020    4.435908e+10  4.279751e+09  3.292742e+09 -2.368176e+10   
...                     ...           ...           ...           ...   
VRE    2013             NaN           NaN           NaN           NaN   
VTK    2014             NaN           NaN           NaN           NaN   
VTP    2015             NaN           NaN           NaN           NaN   
VTR    2016             NaN           NaN           NaN           NaN   
VW3    2016             NaN           NaN           NaN           NaN   

               fx_effect_direct  net_cash_flow_direct  cash_begin_direct  \
symbol period                                                             

In [267]:
panel.loc[(panel["d_paid_in_capital"] == 0) & (panel["stock_issuance_proceeds"].isna()), "stock_issuance_proceeds"] = 0

In [268]:
panel["stock_issuance_proceeds"]

symbol  period
A32     2016      NaN
        2017      0.0
        2018      0.0
        2019      0.0
        2020      0.0
                 ... 
VTR     2016      NaN
VW3     2016      NaN
VXP     2015      0.0
X18     2010      0.0
VEE     2009      0.0
Name: stock_issuance_proceeds, Length: 23609, dtype: float64

In [269]:
panel[panel["stock_issuance_proceeds"].isna()]

owner_equity  minority_interest  paid_in_capital  \
symbol period                                                     
A32    2016    1.628425e+11                0.0     6.800000e+10   
AAT    2018    3.897480e+11                0.0     3.480000e+11   
       2021    6.687368e+11                0.0     6.380149e+11   
       2023    7.076211e+11                0.0     7.081910e+11   
ABR    2017    3.389804e+10        230988574.0     3.000000e+10   
...                     ...                ...              ...   
VRE    2013             NaN                NaN              NaN   
VTK    2014             NaN                NaN              NaN   
VTP    2015             NaN                NaN              NaN   
VTR    2016             NaN                NaN              NaN   
VW3    2016             NaN                NaN              NaN   

               treasury_stock  total_assets  long_term_debt  current_assets  \
symbol period                                                                 
A32    2016               0.0  4.650048e+11    2.429378e+09    3.432787e+11   
AAT    2018               0.0  5.932450e+11    1.003708e+11    1.288567e+11   
       2021               0.0  9.766124e+11    1.384863e+11    3.432234e+11   
       2023               0.0  1.011341e+12    7.577043e+10    4.680603e+11   
ABR    2017               0.0  3.862987e+10    0.000000e+00    2.695623e+10   
...                       ...           ...             ...             ...   
VRE    2013               NaN           NaN             NaN             NaN   
VTK    2014               NaN           NaN             NaN             NaN   
VTP    2015               NaN           NaN             NaN             NaN   
VTR    2016               NaN           NaN             NaN             NaN   
VW3    2016               NaN           NaN             NaN             NaN   

               current_liabilities  total_liabilities  equity_section  ...  \
symbol period                                                          ...   
A32    2016           2.991735e+11       3.016029e+11    1.634019e+11  ...   
AAT    2018           8.393537e+10       2.034971e+11    3.897480e+11  ...   
       2021           1.157017e+11       3.078756e+11    6.687368e+11  ...   
       2023           2.279495e+11       3.037199e+11    7.076211e+11  ...   
ABR    2017           4.729665e+09       4.731831e+09    3.389804e+10  ...   
...                            ...                ...             ...  ...   
VRE    2013                    NaN                NaN             NaN  ...   
VTK    2014                    NaN                NaN             NaN  ...   
VTP    2015                    NaN                NaN             NaN  ...   
VTR    2016                    NaN                NaN             NaN  ...   
VW3    2016                    NaN                NaN             NaN  ...   

                   cash_end    cfo_direct    cfi_direct    cff_direct  \
symbol period                                                           
A32    2016    1.397735e+11 -3.127942e+10 -8.319863e+09 -9.400540e+10   
AAT    2018    6.716559e+09  7.072310e+10 -2.641104e+10 -3.873912e+10   
       2021    9.331644e+10  4.815005e+10 -2.967509e+11  3.221807e+11   
       2023    2.079219e+09  4.138865e+10 -3.600957e+10 -1.895221e+10   
ABR    2017    4.146754e+09  3.938492e+09 -2.595487e+09  1.602000e+09   
...                     ...           ...           ...           ...   
VRE    2013             NaN           NaN           NaN           NaN   
VTK    2014             NaN           NaN           NaN           NaN   
VTP    2015             NaN           NaN           NaN           NaN   
VTR    2016             NaN           NaN           NaN           NaN   
VW3    2016             NaN           NaN           NaN           NaN   

               fx_effect_direct  net_cash_flow_direct  cash_begin_direct  \
symbol period                                                             

In [270]:
panel = panel[panel.index.get_level_values("symbol").isin(non_financial_firms["symbol"])]
panel

owner_equity  minority_interest  paid_in_capital  \
symbol period                                                     
A32    2016    1.628425e+11                0.0     6.800000e+10   
       2017    1.763000e+11                0.0     6.800000e+10   
       2018    2.008266e+11                0.0     6.800000e+10   
       2019    2.236159e+11                0.0     6.800000e+10   
       2020    2.422229e+11                0.0     6.800000e+10   
...                     ...                ...              ...   
VTR    2016             NaN                NaN              NaN   
VW3    2016             NaN                NaN              NaN   
VXP    2015             NaN                NaN              NaN   
X18    2010             NaN                NaN              NaN   
VEE    2009             NaN                NaN              NaN   

               treasury_stock  total_assets  long_term_debt  current_assets  \
symbol period                                                                 
A32    2016               0.0  4.650048e+11    2.429378e+09    3.432787e+11   
       2017               0.0  5.009923e+11    2.429378e+09    3.741267e+11   
       2018               0.0  4.688411e+11    2.429378e+09    3.358630e+11   
       2019               0.0  4.349303e+11    1.429378e+09    2.987680e+11   
       2020               0.0  4.882955e+11    0.000000e+00    3.566913e+11   
...                       ...           ...             ...             ...   
VTR    2016               NaN           NaN             NaN             NaN   
VW3    2016               NaN           NaN             NaN             NaN   
VXP    2015               NaN           NaN             NaN             NaN   
X18    2010               NaN           NaN             NaN             NaN   
VEE    2009               NaN           NaN             NaN             NaN   

               current_liabilities  total_liabilities  equity_section  ...  \
symbol period                                                          ...   
A32    2016           2.991735e+11       3.016029e+11    1.634019e+11  ...   
       2017           3.222796e+11       3.247090e+11    1.762833e+11  ...   
       2018           2.656000e+11       2.680293e+11    2.008118e+11  ...   
       2019           2.098882e+11       2.113176e+11    2.236127e+11  ...   
       2020           2.460785e+11       2.460785e+11    2.422169e+11  ...   
...                            ...                ...             ...  ...   
VTR    2016                    NaN                NaN             NaN  ...   
VW3    2016                    NaN                NaN             NaN  ...   
VXP    2015                    NaN                NaN             NaN  ...   
X18    2010                    NaN                NaN             NaN  ...   
VEE    2009                    NaN                NaN             NaN  ...   

                   cash_end    cfo_direct    cfi_direct    cff_direct  \
symbol period                                                           
A32    2016    1.397735e+11 -3.127942e+10 -8.319863e+09 -9.400540e+10   
       2017    1.456583e+11 -6.679452e+09 -2.375775e+09  1.494000e+10   
       2018    5.829081e+10  5.224003e+10  3.452162e+09 -1.430597e+11   
       2019    6.051375e+10  1.120983e+10  9.204970e+08 -9.907377e+09   
       2020    4.435908e+10  4.279751e+09  3.292742e+09 -2.368176e+10   
...                     ...           ...           ...           ...   
VTR    2016             NaN           NaN           NaN           NaN   
VW3    2016             NaN           NaN           NaN           NaN   
VXP    2015    2.629632e+10           NaN           NaN           NaN   
X18    2010    5.167372e+09           NaN           NaN           NaN   
VEE    2009    1.218471e+10           NaN           NaN           NaN   

               fx_effect_direct  net_cash_flow_direct  cash_begin_direct  \
symbol period                                                             

In [271]:
panel[panel["stock_issuance_proceeds"].isna()]

owner_equity  minority_interest  paid_in_capital  \
symbol period                                                     
A32    2016    1.628425e+11                0.0     6.800000e+10   
AAT    2018    3.897480e+11                0.0     3.480000e+11   
       2021    6.687368e+11                0.0     6.380149e+11   
       2023    7.076211e+11                0.0     7.081910e+11   
ABR    2017    3.389804e+10        230988574.0     3.000000e+10   
...                     ...                ...              ...   
VRE    2013             NaN                NaN              NaN   
VTK    2014             NaN                NaN              NaN   
VTP    2015             NaN                NaN              NaN   
VTR    2016             NaN                NaN              NaN   
VW3    2016             NaN                NaN              NaN   

               treasury_stock  total_assets  long_term_debt  current_assets  \
symbol period                                                                 
A32    2016               0.0  4.650048e+11    2.429378e+09    3.432787e+11   
AAT    2018               0.0  5.932450e+11    1.003708e+11    1.288567e+11   
       2021               0.0  9.766124e+11    1.384863e+11    3.432234e+11   
       2023               0.0  1.011341e+12    7.577043e+10    4.680603e+11   
ABR    2017               0.0  3.862987e+10    0.000000e+00    2.695623e+10   
...                       ...           ...             ...             ...   
VRE    2013               NaN           NaN             NaN             NaN   
VTK    2014               NaN           NaN             NaN             NaN   
VTP    2015               NaN           NaN             NaN             NaN   
VTR    2016               NaN           NaN             NaN             NaN   
VW3    2016               NaN           NaN             NaN             NaN   

               current_liabilities  total_liabilities  equity_section  ...  \
symbol period                                                          ...   
A32    2016           2.991735e+11       3.016029e+11    1.634019e+11  ...   
AAT    2018           8.393537e+10       2.034971e+11    3.897480e+11  ...   
       2021           1.157017e+11       3.078756e+11    6.687368e+11  ...   
       2023           2.279495e+11       3.037199e+11    7.076211e+11  ...   
ABR    2017           4.729665e+09       4.731831e+09    3.389804e+10  ...   
...                            ...                ...             ...  ...   
VRE    2013                    NaN                NaN             NaN  ...   
VTK    2014                    NaN                NaN             NaN  ...   
VTP    2015                    NaN                NaN             NaN  ...   
VTR    2016                    NaN                NaN             NaN  ...   
VW3    2016                    NaN                NaN             NaN  ...   

                   cash_end    cfo_direct    cfi_direct    cff_direct  \
symbol period                                                           
A32    2016    1.397735e+11 -3.127942e+10 -8.319863e+09 -9.400540e+10   
AAT    2018    6.716559e+09  7.072310e+10 -2.641104e+10 -3.873912e+10   
       2021    9.331644e+10  4.815005e+10 -2.967509e+11  3.221807e+11   
       2023    2.079219e+09  4.138865e+10 -3.600957e+10 -1.895221e+10   
ABR    2017    4.146754e+09  3.938492e+09 -2.595487e+09  1.602000e+09   
...                     ...           ...           ...           ...   
VRE    2013             NaN           NaN           NaN           NaN   
VTK    2014             NaN           NaN           NaN           NaN   
VTP    2015             NaN           NaN           NaN           NaN   
VTR    2016             NaN           NaN           NaN           NaN   
VW3    2016             NaN           NaN           NaN           NaN   

               fx_effect_direct  net_cash_flow_direct  cash_begin_direct  \
symbol period                                                             

In [250]:
panel.columns

Index(['owner_equity', 'minority_interest', 'paid_in_capital',
       'treasury_stock', 'total_assets', 'long_term_debt', 'current_assets',
       'current_liabilities', 'total_liabilities', 'equity_section',
       'long_term_liabilities', 'cash_and_equivalents', 'net_income_parent',
       'gross_profit', 'net_sales', 'operating_profit', 'other_profit',
       'pretax_profit', 'cogs', 'net_income_total', 'net_income_minority',
       'cfo', 'stock_issuance_proceeds', 'cfi', 'cff', 'fx_effect',
       'net_cash_flow', 'cash_begin', 'cash_end', 'cfo_direct', 'cfi_direct',
       'cff_direct', 'fx_effect_direct', 'net_cash_flow_direct',
       'cash_begin_direct', 'cash_end_direct', 'paid_in_capital_prev',
       'd_paid_in_capital'],
      dtype='str')

In [272]:
panel.loc[("HPG", "2025"), "paid_in_capital"]

np.float64(76754658550000.0)

In [273]:
panel.to_csv("../data/preprocessing_pipeline_results/f_score_fields_extract.csv")

In [253]:
REQ = [k for _, k in sf.REQUIRED]
coverage_df = panel[REQ].notna()

coverage_df

owner_equity  minority_interest  paid_in_capital  \
symbol period                                                     
A32    2016            True               True             True   
       2017            True               True             True   
       2018            True               True             True   
       2019            True               True             True   
       2020            True               True             True   
...                     ...                ...              ...   
VTK    2014           False              False            False   
VTP    2015           False              False            False   
VTR    2016           False              False            False   
VW3    2016           False              False            False   
VXP    2015           False              False            False   

               treasury_stock  net_income_parent  total_assets    cfo  \
symbol period                                                           
A32    2016              True               True          True   True   
       2017              True               True          True   True   
       2018              True               True          True   True   
       2019              True               True          True   True   
       2020              True               True          True   True   
...                       ...                ...           ...    ...   
VTK    2014             False               True         False  False   
VTP    2015             False               True         False  False   
VTR    2016             False               True         False  False   
VW3    2016             False               True         False  False   
VXP    2015             False               True         False   True   

               long_term_debt  current_assets  current_liabilities  ...  \
symbol period                                                       ...   
A32    2016              True            True                 True  ...   
       2017              True            True                 True  ...   
       2018              True            True                 True  ...   
       2019              True            True                 True  ...   
       2020              True            True                 True  ...   
...                       ...             ...                  ...  ...   
VTK    2014             False           False                False  ...   
VTP    2015             False           False                False  ...   
VTR    2016             False           False                False  ...   
VW3    2016             False           False                False  ...   
VXP    2015             False           False                False  ...   

               net_cash_flow  cash_begin  cash_end  cfo_direct  cfi_direct  \
symbol period                                                                
A32    2016             True        True      True        True        True   
       2017             True        True      True        True        True   
       2018             True        True      True        True        True   
       2019             True        True      True        True        True   
       2020             True        True      True        True        True   
...                      ...         ...       ...         ...         ...   
VTK    2014            False       False     False       False       False   
VTP    2015            False       False     False       False       False   
VTR    2016            False       False     False       False       False   
VW3    2016            False       False     False       False       False   
VXP    2015             True        True      True       False       False   

               cff_direct  fx_effect_direct  net_cash_flow_direct  \
symbol period                                                       
A32    2016          True              True                  Tru

In [254]:
coverage_df.groupby(by="symbol", sort=False).mean()

,owner_equity,minority_interest,paid_in_capital,treasury_stock,net_income_parent,total_assets,cfo,long_term_debt,current_assets,current_liabilities,...,net_cash_flow,cash_begin,cash_end,cfo_direct,cfi_direct,cff_direct,fx_effect_direct,net_cash_flow_direct,cash_begin_direct,cash_end_direct
symbol,,,,,,,,,,,,,,,,,,,,,
A32,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.000000,1.000000,1.000000,0.900,1.000000,1.000000,1.000000
AAA,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000
AAH,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,0.142857,0.142857,0.142857,0.000,0.142857,0.142857,0.142857
AAM,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000
AAT,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,0.750000,0.750000,0.750000,0.625,0.750000,0.750000,0.750000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XPH,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000
YBC,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000
YBM,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000


In [255]:
coverage_df
#Todo
#1. Turnover dist
#2. Internal consistency
#3. Generalize the code so that can run on cafef
#4. External consistency

owner_equity  minority_interest  paid_in_capital  \
symbol period                                                     
A32    2016            True               True             True   
       2017            True               True             True   
       2018            True               True             True   
       2019            True               True             True   
       2020            True               True             True   
...                     ...                ...              ...   
VTK    2014           False              False            False   
VTP    2015           False              False            False   
VTR    2016           False              False            False   
VW3    2016           False              False            False   
VXP    2015           False              False            False   

               treasury_stock  net_income_parent  total_assets    cfo  \
symbol period                                                           
A32    2016              True               True          True   True   
       2017              True               True          True   True   
       2018              True               True          True   True   
       2019              True               True          True   True   
       2020              True               True          True   True   
...                       ...                ...           ...    ...   
VTK    2014             False               True         False  False   
VTP    2015             False               True         False  False   
VTR    2016             False               True         False  False   
VW3    2016             False               True         False  False   
VXP    2015             False               True         False   True   

               long_term_debt  current_assets  current_liabilities  ...  \
symbol period                                                       ...   
A32    2016              True            True                 True  ...   
       2017              True            True                 True  ...   
       2018              True            True                 True  ...   
       2019              True            True                 True  ...   
       2020              True            True                 True  ...   
...                       ...             ...                  ...  ...   
VTK    2014             False           False                False  ...   
VTP    2015             False           False                False  ...   
VTR    2016             False           False                False  ...   
VW3    2016             False           False                False  ...   
VXP    2015             False           False                False  ...   

               net_cash_flow  cash_begin  cash_end  cfo_direct  cfi_direct  \
symbol period                                                                
A32    2016             True        True      True        True        True   
       2017             True        True      True        True        True   
       2018             True        True      True        True        True   
       2019             True        True      True        True        True   
       2020             True        True      True        True        True   
...                      ...         ...       ...         ...         ...   
VTK    2014            False       False     False       False       False   
VTP    2015            False       False     False       False       False   
VTR    2016            False       False     False       False       False   
VW3    2016            False       False     False       False       False   
VXP    2015             True        True      True       False       False   

               cff_direct  fx_effect_direct  net_cash_flow_direct  \
symbol period                                                       
A32    2016          True              True                  Tru